# Feature Engineering — Thời gian & Lag

| Feature | Mô tả |
|---|---|
| `year`, `month`, `day` | Vị trí lịch cơ bản |
| `day_of_week`, `week_of_year`, `quarter` | Chu kỳ tuần / năm |
| `is_weekend`, `is_month_start`, `is_month_end`, `is_payday` | Flag đặc biệt Ecuador |
| `lag_1/2/3`, `lag_7`, `lag_14/28/364` | Lag doanh số (nhóm store-family) |
| `rolling_mean_7/14/28`, `rolling_std_7` | Rolling stats (shift-1 safe, không leakage) |

In [1]:
import pandas as pd
import numpy as np

In [ ]:
import sys
from pathlib import Path

def _find_root():
    """Tìm project root chứa config.py."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'config.py').exists():
            return p
    raise RuntimeError("Không tìm thấy project root. Mở Jupyter từ thư mục gốc của project.")

_root = _find_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import PROCESSED_DIR

df = pd.read_csv(PROCESSED_DIR / 'train_cleaned.csv')

print(f'Shape   : {df.shape}')
print(f'Columns : {df.columns.tolist()}')

---
## 1. Temporal Features

In [3]:
df = df.copy()
df['date'] = pd.to_datetime(df['date'])

dt = df['date'].dt

df['year']         = dt.year
df['month']        = dt.month
df['day']          = dt.day
df['day_of_week']  = dt.dayofweek           # 0 = Monday, 6 = Sunday
df['week_of_year'] = dt.isocalendar().week.astype(int)
df['quarter']      = dt.quarter
df['is_weekend']   = (dt.dayofweek >= 5).astype(int)
df['is_month_start'] = dt.is_month_start.astype(int)
df['is_month_end']   = dt.is_month_end.astype(int)
df['is_payday']    = ((dt.day == 15) | dt.is_month_end).astype(int)

print('Calendar units and binary flags added:')
df[['date', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 
    'is_weekend', 'is_month_start', 'is_month_end', 'is_payday']].head(5)


Calendar units and binary flags added:


,date,year,month,day,day_of_week,week_of_year,quarter,is_weekend,is_month_start,is_month_end,is_payday
0,2013-01-01,2013,1,1,1,1,1,0,1,0,0
1,2013-01-01,2013,1,1,1,1,1,0,1,0,0
2,2013-01-01,2013,1,1,1,1,1,0,1,0,0
3,2013-01-01,2013,1,1,1,1,1,0,1,0,0
4,2013-01-01,2013,1,1,1,1,1,0,1,0,0


In [4]:
TEMPORAL_FEATURE_NAMES = [
    'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter',
    'is_weekend', 'is_month_start', 'is_month_end', 'is_payday',
]

print('Temporal feature check:')
summary = df[TEMPORAL_FEATURE_NAMES].agg(['min', 'max', lambda x: x.isna().sum()]).T
summary.columns = ['min', 'max', 'null_count']
print(summary)

Temporal feature check:
                 min   max  null_count
year            2013  2017           0
month              1    12           0
day                1    31           0
day_of_week        0     6           0
week_of_year       1    53           0
quarter            1     4           0
is_weekend         0     1           0
is_month_start     0     1           0
is_month_end       0     1           0
is_payday          0     1           0


---
## 2. Lag & Rolling Features

Tất cả lag/rolling dùng `shift(1)` — window cho dòng `t` chỉ bao gồm `[t-w, ..., t-1]`, không leakage.

In [5]:
GROUP_COLS  = ['store_nbr', 'family']
TARGET_COL  = 'sales'
DATE_COL    = 'date'

df = df.sort_values(GROUP_COLS + [DATE_COL]).reset_index(drop=True)

grouped = df.groupby(GROUP_COLS, sort=False)[TARGET_COL]

print(f'Sorted by: {GROUP_COLS + [DATE_COL]}')
print(f'Number of time series (store × family): {df.groupby(GROUP_COLS).ngroups}')
df[GROUP_COLS + [DATE_COL, TARGET_COL]].head(6)

Sorted by: ['store_nbr', 'family', 'date']
Number of time series (store × family): 1782


,store_nbr,family,date,sales
0,1,AUTOMOTIVE,2013-01-01,0.0
1,1,AUTOMOTIVE,2013-01-02,2.0
2,1,AUTOMOTIVE,2013-01-03,3.0
3,1,AUTOMOTIVE,2013-01-04,3.0
4,1,AUTOMOTIVE,2013-01-05,5.0
5,1,AUTOMOTIVE,2013-01-06,2.0


### Lag ngắn hạn (1, 2, 3 ngày)

In [6]:
for lag in [1, 2, 3]:
    df[f'lag_{lag}'] = grouped.shift(lag)

print('Short-term lags added. NaN counts (expected at group boundaries):')
df[['lag_1', 'lag_2', 'lag_3']].isna().sum()

Short-term lags added. NaN counts (expected at group boundaries):


lag_1    1782
lag_2    3564
lag_3    5346
dtype: int64

In [7]:
df['lag_7'] = grouped.shift(7)

print(f'lag_7 NaN count : {df["lag_7"].isna().sum()}')
print(f'lag_7 coverage  : {(1 - df["lag_7"].isna().mean()) * 100:.1f}%')

lag_7 NaN count : 12474
lag_7 coverage  : 99.6%


### Lag dài hạn (14, 28, 364 ngày)

In [8]:
for lag in [14, 28, 364]:
    df[f'lag_{lag}'] = grouped.shift(lag)

long_lag_cols = ['lag_14', 'lag_28', 'lag_364']
print('Longer lag NaN counts and coverage:')
for col in long_lag_cols:
    coverage = (1 - df[col].isna().mean()) * 100
    print(f'  {col:<10}  NaN={df[col].isna().sum():>6}   coverage={coverage:.1f}%')

Longer lag NaN counts and coverage:
  lag_14      NaN= 24948   coverage=99.2%
  lag_28      NaN= 49896   coverage=98.3%
  lag_364     NaN=648648   coverage=78.4%


### Rolling Stats (7/14/28 ngày)

In [9]:
# shift(1) guarantees the rolling window never includes the current observation
shifted = grouped.shift(1)

df['rolling_mean_7']  = shifted.transform(lambda x: x.rolling(7,  min_periods=1).mean())
df['rolling_mean_14'] = shifted.transform(lambda x: x.rolling(14, min_periods=1).mean())
df['rolling_mean_28'] = shifted.transform(lambda x: x.rolling(28, min_periods=1).mean())
df['rolling_std_7']   = shifted.transform(lambda x: x.rolling(7,  min_periods=2).std())

rolling_cols = ['rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7']
print('Rolling feature summary:')
df[rolling_cols].describe().T[['mean', 'std', 'min', 'max']]

Rolling feature summary:


,mean,std,min,max
rolling_mean_7,357.704680,1047.254096,0.0,29937.285714
rolling_mean_14,357.711654,1041.093101,0.0,20097.642857
rolling_mean_28,357.717832,1034.296458,0.0,16118.321429
rolling_std_7,106.522145,354.699813,0.0,47275.358965


---
## 3. Pipeline Tổng hợp

In [ ]:
def add_temporal_features(df: pd.DataFrame, date_col: str = 'date') -> pd.DataFrame:
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    dt = df[date_col].dt

    df['year']           = dt.year
    df['month']          = dt.month
    df['day']            = dt.day
    df['day_of_week']    = dt.dayofweek
    df['week_of_year']   = dt.isocalendar().week.astype(int)
    df['quarter']        = dt.quarter
    df['is_weekend']     = (dt.dayofweek >= 5).astype(int)
    df['is_month_start'] = dt.is_month_start.astype(int)
    df['is_month_end']   = dt.is_month_end.astype(int)
    df['is_payday']      = ((dt.day == 15) | dt.is_month_end).astype(int)
    return df


def add_lag_features(
    df: pd.DataFrame,
    target_col: str = 'sales',
    group_cols: list = None,
    date_col: str = 'date',
) -> pd.DataFrame:
    if group_cols is None:
        group_cols = ['store_nbr', 'family']

    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(group_cols + [date_col]).reset_index(drop=True)

    grouped = df.groupby(group_cols, sort=False)[target_col]

    for lag in [1, 2, 3]:
        df[f'lag_{lag}'] = grouped.shift(lag)
    df['lag_7'] = grouped.shift(7)
    for lag in [14, 28, 364]:
        df[f'lag_{lag}'] = grouped.shift(lag)

    shifted = grouped.shift(1)
    df['rolling_mean_7']  = shifted.transform(lambda x: x.rolling(7,  min_periods=1).mean())
    df['rolling_mean_14'] = shifted.transform(lambda x: x.rolling(14, min_periods=1).mean())
    df['rolling_mean_28'] = shifted.transform(lambda x: x.rolling(28, min_periods=1).mean())
    df['rolling_std_7']   = shifted.transform(lambda x: x.rolling(7,  min_periods=2).std())
    return df


def build_features(
    df: pd.DataFrame,
    target_col: str = 'sales',
    group_cols: list = None,
    date_col: str = 'date',
) -> pd.DataFrame:
    df = add_temporal_features(df, date_col=date_col)
    df = add_lag_features(df, target_col=target_col, group_cols=group_cols, date_col=date_col)
    return df


raw = pd.read_csv(PROCESSED_DIR / 'train_cleaned.csv')
df_featured = build_features(raw)

print(f'Input shape  : {raw.shape}')
print(f'Output shape : {df_featured.shape}')
print(f'New columns  : {df_featured.shape[1] - raw.shape[1]}')

---
## 4. Feature Name Registry

In [11]:
TEMPORAL_FEATURE_NAMES = [
    'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter',
    'is_weekend', 'is_month_start', 'is_month_end', 'is_payday',
]

LAG_FEATURE_NAMES = [
    'lag_1', 'lag_2', 'lag_3',
    'lag_7',
    'lag_14', 'lag_28', 'lag_364',
    'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28',
    'rolling_std_7',
]

ALL_FEATURE_NAMES = TEMPORAL_FEATURE_NAMES + LAG_FEATURE_NAMES

print(f'Temporal features : {len(TEMPORAL_FEATURE_NAMES)}')
print(f'Lag features      : {len(LAG_FEATURE_NAMES)}')
print(f'Total features    : {len(ALL_FEATURE_NAMES)}')
print(f'\nAll features: {ALL_FEATURE_NAMES}')

Temporal features : 10
Lag features      : 11
Total features    : 21

All features: ['year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'is_payday', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_28', 'lag_364', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7']


---
## 5. Xác thực & Anti-leakage

### Smoke Test (2 stores × 60 ngày)

In [12]:
# --- Build mock data ---
dates = pd.date_range('2017-01-01', periods=60, freq='D')
mock = pd.DataFrame({
    'date'     : list(dates) * 2,
    'store_nbr': [1] * 60 + [2] * 60,
    'family'   : ['GROCERY I'] * 120,
    'sales'    : np.random.rand(120) * 1000,
})

# --- Run pipeline ---
result = build_features(mock)

print(f'Mock input shape  : {mock.shape}')
print(f'Mock output shape : {result.shape}')

Mock input shape  : (120, 4)
Mock output shape : (120, 25)


### Assertion: lag_1 dòng đầu mỗi group phải = NaN

In [ ]:
# groupby().nth(0) lấy dòng đầu tiên thực sự (có thể NaN),
# khác .first() trả về giá trị non-null đầu tiên → false positive.
first_rows = (
    result
    .sort_values(['store_nbr', 'family', 'date'])
    .groupby(['store_nbr', 'family'])
    .nth(0)
)

bad = first_rows[~first_rows['lag_1'].isna()]
if not bad.empty:
    print("Data leakage detected:")
    print(bad[['lag_1']])
    raise AssertionError("lag_1 của dòng đầu tiên mỗi group phải là NaN.")

print('Anti-leakage check passed ✓')
print(first_rows[['lag_1']])

In [ ]:
# CSV write disabled — tất cả features được merge trong build_train_final.py.
# Để tạo CSV cho model training:
#   python scripts/build_train_final.py   →  data/processed/train_final.csv
#   python scripts/create_splits.py       →  data/processed/splits/
print('[SKIPPED] CSV write disabled')
print(f'Feature dataframe shape : {df_featured.shape}')
print(f'Columns: {df_featured.columns.tolist()}')